### 1. Data Cleaning and Preprocessing

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Read the original dataset
df = pd.read_csv('phishing_email.csv')
print(f"dataset shape: {df.shape}")
# Drop duplicates and rows with missing labels or text
df = df.dropna(subset=['text_combined', 'label'])
df = df.drop_duplicates()

# Ensure labels are of integer type
df['label'] = df['label'].astype(int)

print(f"Cleaned dataset shape: {df.shape}")

dataset shape: (82486, 2)
Cleaned dataset shape: (82078, 2)


### 2. Splitting the Dataset (60% Train, 20% Validation, 20% Test)

In [3]:
# Train-Test Split (80/20) on the full dataset
df_train_val, df_test = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])

# Train-Validation Split (75/25 of train_val => 60/20/20 of total)
df_train, df_validation = train_test_split(df_train_val, test_size=0.25, random_state=42, stratify=df_train_val['label'])

### 3. Stage 1: TF-IDF Vectorizer & First Logistic Regression Model

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import pandas as pd

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit and transform datasets
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train['text_combined'])
X_validation_tfidf = tfidf_vectorizer.transform(df_validation['text_combined'])
X_test_tfidf = tfidf_vectorizer.transform(df_test['text_combined'])

y_train = df_train['label']
y_validation = df_validation['label']
y_test = df_test['label']

# Train the Stage 1 Classifier
stage1_model = LogisticRegression(max_iter=1000, random_state=42)
stage1_model.fit(X_train_tfidf, y_train)

# --- Evaluate Stage 1 on the Test Set ---
y_test_pred_stage1 = stage1_model.predict(X_test_tfidf)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_test_pred_stage1, average='binary')
accuracy = accuracy_score(y_test, y_test_pred_stage1)


print("\nStage 1 Classification Report:")
print(classification_report(y_test, y_test_pred_stage1))

# --- Extract low-confidence test samples for Stage 2 ---
y_test_pred_proba = stage1_model.predict_proba(X_test_tfidf)[:, 1]
confidence_threshold = 0.80

# Low confidence means the probability is not strongly 0 or 1 (< 0.80)
low_confidence_mask = (y_test_pred_proba < confidence_threshold)
df_test_low_conf = df_test[low_confidence_mask].copy()


print(f"High Confidence (Handled by Stage 1): {len(df_test) - len(df_test_low_confidence)}")
print(f"Low Confidence (To be routed to Stage 2): {len(df_test_low_confidence)}")


Stage 1 Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      7847
           1       0.98      0.98      0.98      8569

    accuracy                           0.98     16416
   macro avg       0.98      0.98      0.98     16416
weighted avg       0.98      0.98      0.98     16416

High Confidence (Handled by Stage 1): 7909
Low Confidence (To be routed to Stage 2): 8507


### 4. Stage 2: Sentence Transformers & Second Logistic Regression Model

In [ ]:
from sentence_transformers import SentenceTransformer

# Initialize the Sentence Transformer model
#embedding_model = SentenceTransformer('all-MiniLM-L6-v2', local_files_only=True)

# Separate by label
df_low_conf_0 = df_test_low_conf[df_test_low_conf['label'] == 0]
df_low_conf_1 = df_test_low_conf[df_test_low_conf['label'] == 1]

# Randomly sample 500 rows from each class (total 1k balanced training samples)
sampled_0 = df_low_conf_0.sample(n=500, random_state=42)
sampled_1 = df_low_conf_1.sample(n=500, random_state=42)

# Combine to make our final 1k training sample for Stage 2
df_train_stage2_sampled = pd.concat([sampled_0, sampled_1]).sample(frac=1, random_state=42)

# Identify remaining rows for testing Stage 2 locally
df_test_stage2_remaining = df_test_low_conf.drop(df_train_stage2_sampled.index)


#Extract embeddings for the balanced training subset
print("\nEncoding 1k balanced training samples...")
X_train_embed = embedding_model.encode(
    df_train_stage2_sampled['text_combined'].tolist(),
    batch_size=128,
    show_progress_bar=True
)
y_train_low_conf = df_train_stage2_sampled['label']

# Train Stage 2 Classifier
stage2_model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
stage2_model.fit(X_train_embed, y_train_low_conf)


Original Low-Confidence Test Size: 8507
Selected Balanced Training Subset Size: 1000 (500 class 0, 500 class 1)
Remaining Evaluation Set Size (Stage 2 Test): 7507

Encoding 1k balanced training samples...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches: 100%|██████████| 8/8 [05:13<00:00, 39.16s/it]

Stage 2 Embeddings Model trained successfully on balanced 1k subset!


In [22]:
# Encode and evaluate on the remaining low-confidence set to see how it generalizes
print("\nEncoding remaining low-confidence samples for evaluation...")
X_remaining_embed = embedding_model.encode(
    df_test_stage2_remaining['text_combined'].tolist(),
    batch_size=128,
    show_progress_bar=True
)
y_remaining_true = df_test_stage2_remaining['label']
y_remaining_pred = stage2_model.predict(X_remaining_embed)

print(f"Accuracy: {accuracy_score(y_remaining_true, y_remaining_pred):.4f}")
print(classification_report(y_remaining_true, y_remaining_pred))


Encoding remaining low-confidence samples for evaluation...


Batches: 100%|██████████| 59/59 [36:35<00:00, 37.22s/it]


Accuracy: 0.8661
              precision    recall  f1-score   support

           0       1.00      0.87      0.93      7305
           1       0.15      0.87      0.26       202

    accuracy                           0.87      7507
   macro avg       0.57      0.87      0.59      7507
weighted avg       0.97      0.87      0.91      7507



### 5. Dual-Stage Inference Pipeline Function

In [23]:
def predict_hybrid_pipeline(dataframe, confidence_threshold=0.80):
    """
    Evaluates a dataframe through the two-stage model workflow:
    - Stage 1: TF-IDF + Logistic Regression.
    - Samples classified as 1 with probability >= confidence_threshold are predicted as 1.
    - All remaining samples are passed to Stage 2 (Sentence Transformer embeddings + Logistic Regression).
    """
    results_df = dataframe.copy()

    # Stage 1 prediction probabilities
    X_tfidf = tfidf_vectorizer.transform(results_df['text_combined'])
    stage1_probs = stage1_model.predict_proba(X_tfidf)[:, 1]

    results_df['stage1_prob'] = stage1_probs

    final_predictions = np.zeros(len(results_df), dtype=int)
    routing_decision = []

    low_conf_indices = []
    for i, prob in enumerate(stage1_probs):
        if prob >= confidence_threshold:
            final_predictions[i] = 1
            routing_decision.append('Stage 1 (High Confidence)')
        else:
            low_conf_indices.append(i)
            routing_decision.append('Stage 2 (Embeddings)')

    # If there are samples assigned to Stage 2, run them through the second model with GPU optimization
    if len(low_conf_indices) > 0:
        low_conf_texts = results_df.iloc[low_conf_indices]['text_combined'].tolist()
        # Run encoding on GPU if configured
        X_embed = embedding_model.encode(low_conf_texts, batch_size=128, show_progress_bar=False)
        stage2_preds = stage2_model.predict(X_embed)

        for local_idx, global_idx in enumerate(low_conf_indices):
            final_predictions[global_idx] = stage2_preds[local_idx]

    results_df['predicted_label'] = final_predictions
    results_df['pipeline_route'] = routing_decision

    return results_df

### 6. Test on the Validation Set

In [24]:
# Run the complete hybrid pipeline on the full test set
test_results = predict_hybrid_pipeline(df_test, confidence_threshold=0.80)

In [26]:
print("Test Set Classification Report:")
print(classification_report(test_results['label'], test_results['predicted_label']))

# Display how the test samples were routed between the models
route_counts = test_results['pipeline_route'].value_counts()
print("=== Test Set Routing Breakdown ===")
for route, count in route_counts.items():
    pct = (count / len(test_results)) * 100
    print(f"{route}: {count} samples ({pct:.2f}%)")

Test Set Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.86      0.92      7847
           1       0.89      0.99      0.94      8569

    accuracy                           0.93     16416
   macro avg       0.94      0.93      0.93     16416
weighted avg       0.94      0.93      0.93     16416

=== Test Set Routing Breakdown ===
Stage 2 (Embeddings): 8507 samples (51.82%)
Stage 1 (High Confidence): 7909 samples (48.18%)


In [28]:
test_results.head()

,text_combined,label,stage1_prob,predicted_label,pipeline_route
30333,quichua info hi quito march 23 april 2 would l...,0,0.120474,0,Stage 2 (Embeddings)
20421,cialis xanax valium viagra low price prescript...,1,0.996550,1,Stage 1 (High Confidence)
55121,jim lemon jimbitwritcomau wed 20080806 1657 07...,0,0.023078,0,Stage 2 (Embeddings)
25965,fw socal rls peaking tariff fyi kim original m...,0,0.150807,0,Stage 2 (Embeddings)
68673,cnn alerts thienapontijnbummde cnn alerts cust...,1,0.980098,1,Stage 1 (High Confidence)
